# FOM Phase 0 -- learned routability residual feasibility study

This notebook is the analysis frontend for **issue #3187**: *does any learnable signal exist between cheap placement features and binary manufacturability?*

It loads the labelled corpus produced by `scripts/research/generate_perturbations.py`, runs leave-one-seed-out cross validation with a `HistGradientBoostingClassifier`, and produces the AUC, calibration, feature-importance, and per-seed performance artefacts that drive the recommendation in `docs/research/learned_fom_phase0.md`.

**All heavy lifting lives in `scripts/research/train_phase0_classifier.py`** so the same code runs from a notebook or a CLI. The notebook just visualises and narrates.

In [ ]:
import json

# Project imports
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))
from kicad_tools.optim.fom_features import PHASE0_FEATURE_NAMES

sys.path.insert(0, str(REPO_ROOT / "scripts" / "research"))
import train_phase0_classifier as t0

warnings.filterwarnings("ignore", category=UserWarning)
print("feature count:", len(PHASE0_FEATURE_NAMES))

## 1. Load the corpus

Each row is one perturbed placement.  `label = 1` iff routing finished *and* `kct check` reported zero DRC errors.

In [ ]:
LABELS = REPO_ROOT / "data" / "research" / "fom_phase0" / "labels.jsonl"
df = t0.load_labels_jsonl(LABELS)
print(f"Loaded {len(df)} samples")
print(f"Positive (label=1) rate: {df.label.mean():.1%}")
print()
print("Per-seed counts:")
print(df.groupby("seed_name")[["label"]].agg(["count", "mean"]))

In [ ]:
# Sanity-check the feature columns are populated.
fcols = t0.feature_columns()
df[fcols].describe().T[["mean", "std", "min", "max"]]

## 2. Leave-one-seed-out cross validation

`sklearn.model_selection.GroupKFold` with `groups=seed_name` ensures no seed appears in both training and test folds. This is the spec's no-data-leakage requirement -- within-seed shuffling would let the model memorise "this layout's commit" instead of learning a true manufacturability signal.

In [ ]:
folds, oof_y, oof_pred = t0.cross_validate_across_seeds(df, fcols)
metrics = t0.compute_metrics(folds, oof_y, oof_pred)
decision, rationale = t0.go_iterate_abandon(metrics)
metrics["decision"] = decision
metrics["decision_rationale"] = rationale
print(json.dumps(metrics, indent=2, sort_keys=True))

## 3. Per-fold (per-seed-holdout) performance

A model that generalises across boards should produce similar AUC on every held-out seed.  A large variance suggests one of the seeds has features that look qualitatively different from the rest -- i.e. small corpus + over-specialised classifier.

In [ ]:
per_seed_df = t0.per_seed_performance(folds)
per_seed_df

## 4. Calibration plot

We use the model output as a **multiplicative weight** in the FOM (`predictor^beta`).  This requires the score to be a *calibrated* probability: when the model says 0.8, it should be wrong only 20% of the time.  A heavily uncalibrated model will systematically bias the FOM toward / away from over-confident regions.

In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(5, 5))
if len(set(oof_y)) >= 2:
    prob_true, prob_pred = calibration_curve(oof_y, oof_pred, n_bins=10, strategy="quantile")
    ax.plot(prob_pred, prob_true, "o-", label="model")
ax.plot([0, 1], [0, 1], "--", color="grey", label="ideal")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("OOF calibration")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(oof_pred[oof_y == 1], bins=20, alpha=0.6, label="label=1")
ax.hist(oof_pred[oof_y == 0], bins=20, alpha=0.6, label="label=0")
ax.set_xlabel("Predicted P(manufacturable)")
ax.set_ylabel("Count")
ax.set_title("Predicted-probability distribution")
ax.legend()
plt.show()

## 5. Feature importance (permutation)

Trained on the full corpus, with sklearn's permutation-importance method (which is unbiased for tree models).  If a small number of features dominates, those should probably migrate into #3186's hand-engineered soft-FOM.

In [ ]:
importance_df, full_clf = t0.feature_importance_df(df, fcols)
importance_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
df_sorted = importance_df.sort_values("importance_mean", ascending=True)
ax.barh(df_sorted["feature"], df_sorted["importance_mean"], xerr=df_sorted["importance_std"])
ax.set_xlabel("Permutation importance (mean +/- std)")
ax.set_title("Phase 0 feature importance")
plt.tight_layout()
plt.show()

## 6. Qualitative failure analysis

Pick a handful of samples the model gets confidently wrong and look at the features -- these are the cases we'd want a CNN to see (or extra hand-features to be added).

In [ ]:
fail_df = df.copy()
fail_df["oof_pred"] = np.nan  # default
# Map OOF predictions back to row indices (we ran folds in order)
# Reconstruct using the sample order GroupKFold visited.
# Easiest: refit and predict; not exactly OOF for these specific rows, but
# good enough for qualitative inspection.
X_full = df[fcols].to_numpy(float)
fail_df["full_pred"] = full_clf.predict_proba(X_full)[:, 1]

# False negatives: model says <0.3 but actually labels=1
fn = fail_df[(fail_df.label == 1) & (fail_df.full_pred < 0.3)]
print(f"False negatives (truly routes, model says no): {len(fn)}")
fn[["sample_id", "seed_name", "full_pred"] + fcols[:6]].head(10)

In [ ]:
# False positives: model says >0.7 but actually labels=0
fp = fail_df[(fail_df.label == 0) & (fail_df.full_pred > 0.7)]
print(f"False positives (model says routes, doesnt): {len(fp)}")
fp[["sample_id", "seed_name", "full_pred", "notes"] + fcols[:6]].head(10)

## 7. Decision

Per the issue's go/iterate/abandon rule:

- AUC > 0.70 -> escalate (Phase 1)
- 0.55 < AUC <= 0.70 -> iterate features
- AUC <= 0.55 -> abandon (no signal at this corpus size)

The narrative writeup in `docs/research/learned_fom_phase0.md` reflects the decision below.

In [ ]:
print(f"Decision: {metrics['decision'].upper()}")
print(metrics["decision_rationale"])

## 8. Integration hook (only run if AUC > 0.7)

Demonstrates that the trained classifier plugs into `compute_fom(..., predictor=..., beta=0.1)` from issue #3186 without modification.  When the AUC is too low we skip the demo to keep the artefacts focused on the failure case.

In [ ]:
if (metrics.get("auc_global_oof") or 0.0) > 0.70:
    import joblib

    from kicad_tools.optim.fom import compute_fom
    from kicad_tools.optim.fom_features import extract_phase0_features_from_pcb
    from kicad_tools.schema.pcb import PCB

    CLASSIFIER_PATH = REPO_ROOT / "data" / "research" / "fom_phase0" / "classifier.joblib"
    bundle = joblib.load(CLASSIFIER_PATH)
    estimator = bundle["estimator"]
    feat_names = bundle["feature_names"]

    def predictor(pcb):
        feats = extract_phase0_features_from_pcb(pcb)
        X = np.array([[feats[n] for n in feat_names]], dtype=float)
        return float(estimator.predict_proba(X)[0, 1])

    # Demonstrate on the original board 01 (unrouted).
    pcb = PCB.load(str(REPO_ROOT / "boards/01-voltage-divider/output/voltage_divider.kicad_pcb"))
    p = predictor(pcb)
    print(f"Predicted P(manufacturable) for unrouted board 01 = {p:.3f}")
    res = compute_fom(pcb, predictor=predictor, beta=0.1)
    print(f"FOM with learned residual = {res.score:.4f}")
    print(f"predictor_value = {res.predictor_value:.4f}, beta = {res.beta}")
else:
    print("Skipping integration demo: AUC threshold not met.")